# 18. JOIN Overview

*Module: 5 — Multi-table · Companion summary: `lectures/18-join-overview.md` · SQL engine: SQLite (in-memory)*

## What you will study

- build the borrower ⟕ loan ⟕ copy chain from the course
- connect entities through primary/foreign keys
- traverse three tables in one query

---
## Setup — build the sample database this notebook runs on
*Run the cell below first: it creates an in-memory SQLite database and the `q()` helper that prints every query's result set.*

In [ ]:
import sqlite3
import textwrap

conn = sqlite3.connect(':memory:')  # throw-away in-memory database
conn.execute("PRAGMA foreign_keys = ON")

def q(sql):
    """Run a query, print formatted SQL, and display the result set as a table."""
    clean_sql = textwrap.dedent(sql).strip()

    # 1. Print formatted SQL query
    print("\n" + "=" * 60)
    print("QUERY:")
    print(textwrap.indent(clean_sql, "  "))
    print("-" * 60)

    cur = conn.execute(sql)
    rows = cur.fetchall()

    if cur.description:  # SELECT / statement returning rows
        headers = [d[0] for d in cur.description]
        str_rows = [[("" if v is None else str(v)) for v in r] for r in rows]

        # Calculate dynamic column widths for clean alignment
        widths = [
            max(len(h), *(len(r[i]) for r in str_rows), 1)
            for i, h in enumerate(headers)
        ] if str_rows else [len(h) for h in headers]

        header_line = " | ".join(h.ljust(widths[i]) for i, h in enumerate(headers))
        sep_line = "-+-".join("-" * widths[i] for i in range(len(headers)))

        print(header_line)
        print(sep_line)
        for r in str_rows:
            print(" | ".join(val.ljust(widths[i]) for i, val in enumerate(r)))
        if not str_rows:
            print("(0 rows returned)")
    else:  # DDL / DML (CREATE, INSERT, UPDATE, DELETE)
        affected = cur.rowcount
        print(f"Done (rows affected: {affected})" if affected != -1 else "Done.")

    print("=" * 60)

print('Connection ready:', conn)

## 🧪 Sample data — a tiny library

Three entities linked by keys: BORROWER ←(1:N)— LOAN —(N:1)→ COPY —(N:1)→ BOOK

In [ ]:
q("""
CREATE TABLE book (book_id TEXT PRIMARY KEY, title TEXT, pages INTEGER)
""")
q("""
INSERT INTO book VALUES
    ('B1','The Little Prince', 292),
    ('B2','Journey to the West', 298),
    ('B3','SQL Basics', 150)
""")
q("""
CREATE TABLE copy (
    copy_id TEXT PRIMARY KEY,           -- physical copy of a book
    book_id TEXT REFERENCES book(book_id)
)
""")
q("""
INSERT INTO copy VALUES
    ('C1','B1'), ('C2','B1'), ('C3','B2'), ('C4','B3')
""")
q("""
CREATE TABLE borrower (borrower_id TEXT PRIMARY KEY, lastname TEXT, country TEXT)
""")
q("""
INSERT INTO borrower VALUES
    ('BR1','Smith','CA'), ('BR2','Ali','IN'), ('BR3','Lee','CN'),
    ('BR4','Peters','US'), ('BR5','Li','CN'), ('BR6','Wong','CA')
""")
q("""
CREATE TABLE loan (
    borrower_id TEXT REFERENCES borrower(borrower_id),
    copy_id TEXT REFERENCES copy(copy_id),
    loan_date TEXT
)
""")
q("""
INSERT INTO loan VALUES
    ('BR1','C1','2024-01-10'), ('BR1','C2','2024-02-01'),
    ('BR2','C3','2024-03-05'), ('BR3','C4','2024-04-01')
""")
q("SELECT * FROM book")
q("SELECT * FROM copy")
q("SELECT * FROM borrower")
q("SELECT * FROM loan")

## 🧪 The mental model

A **join** glues tables together by matching key values. The chain for
“who borrowed what”: `borrower → loan → copy → book`, each hop on the shared key.

## 🧪 Two hops: lender ⟕ copy

In [ ]:
q("SELECT b.lastname, l.loan_date, c.copy_id FROM borrower b JOIN loan l ON b.borrower_id = l.borrower_id JOIN copy c ON l.copy_id = c.copy_id")

## 🧪 The full three-table traversal

In [ ]:
q("""
SELECT b.lastname, bk.title, l.loan_date
FROM borrower b
JOIN loan l ON b.borrower_id = l.borrower_id
JOIN copy c ON l.copy_id = c.copy_id
JOIN book bk ON c.book_id = bk.book_id
""")

## 💡 Tips & tricks

- Count your hops: N tables = N−1 join conditions (two forgetting hops = Cartesian
  explosion, nb17).
- Copy entities (physical items) are distinct from catalog entities (books) — a model
  pattern worth remembering.
- 📕 Dialect note: nothing dialect-specific in the join syntax — this is portable SQL.

## 🎯 Your turn

1. Which copy of JOURNEY TO THE WEST is on loan, and to whom?
2. How many loans does each borrower have (group the join result).

In [ ]:
# 💡 Solution
q("""
SELECT c.copy_id, b.lastname
FROM copy c JOIN book bk ON c.book_id = bk.book_id
JOIN loan l ON l.copy_id = c.copy_id
JOIN borrower b ON b.borrower_id = l.borrower_id
WHERE bk.title = 'Journey to the West'
""")
q("""
SELECT b.lastname, COUNT(*) AS n_loans
FROM borrower b JOIN loan l ON b.borrower_id = l.borrower_id
GROUP BY b.lastname
""")